In [ ]:
import uproot, pandas as pd, numpy as np
from pathlib import Path

# ---- Configuration --------------------------------------------------------
BASE_DIR  = Path("PPSSP_2026/1l2tau/run2")   # 1 lepton + 2 taus, Run 2
TREE_NAME = "AnalysisMiniTree"
RUN       = 2

# Preselection cuts for the 1L2Tau channel (see README)
PRESELECTION = "(n_b_jet == 0) & (n_jet >= 2)"

# --- Feature selection policy ---------------------------------------------
# Load generously and let XGBoost prune via importance, BUT never let these
# branches enter as features (they poison the training / leak the label):
#   - weights & scale/fake factors  -> differ systematically signal vs bkg
#   - dsid / eventNumber            -> sample/event identifiers (label in disguise)
#   - *truth* / isTrue* / *fake*    -> generator info, absent in real data
#   - anti-tau bookkeeping          -> fake-region definition, process-dependent
#   - preselection constants        -> n_b_jet==0 always; pass1l2tau==1 always
BLOCK_SUBSTR = ["weight", "effsf", "_ff", "truth", "istrue", "fake", "anti", "dsid", "eventnumber"]
BLOCK_EXACT  = {"n_b_jet", "pass1l2tau", "hhml_subchannelflavor"}

def is_feature(branch: str) -> bool:
    lb = branch.lower()
    return lb not in BLOCK_EXACT and not any(s in lb for s in BLOCK_SUBSTR)

WEIGHT_PARTS = ["weight", "weight_final"]   # raw branches; w_phys = their product

files = {
    # process    : (filename, label)   1 = signal, 0 = background
    "signal_ggF": ("signal_ggF.root", 1),
    "signal_VBF": ("signal_VBF.root", 1),
    "Diboson":    ("diboson.root",    0),
    "Zjets":      ("Zjets.root",      0),
    "Wjets":      ("Wjets.root",      0),
    "ttbar":      ("ttbar.root",      0),
    "tops":       ("tops.root",       0),
    "SingleH":    ("singleH.root",    0),
    "Vgamma":     ("Vgamma.root",     0),
    "VVV":        ("VVV.root",         0),
}

# Build the feature list from branches COMMON to all files (robust against
# per-sample branch differences; a single-file probe could silently misalign)
common = None
for proc, (fname, _) in files.items():
    keys = set(uproot.open({str(BASE_DIR / fname): TREE_NAME}).keys())
    common = keys if common is None else common & keys
features = sorted(b for b in common if is_feature(b))
print(f"{len(features)} candidate features (common to all {len(files)} files, leakage-free)\n")

# ---- Extraction loop ------------------------------------------------------
dfs = []
for proc, (fname, label) in files.items():
    tree = uproot.open({str(BASE_DIR / fname): TREE_NAME})
    df = tree.arrays(features + WEIGHT_PARTS, cut=PRESELECTION, library="pd")
    # 'weight'       = sigma*L/sum_w_gen, per DSID+campaign (raw branch, untouched)
    # 'weight_final' = generator event weight, Sherpa NLO -> negatives (raw branch, untouched)
    # 'w_phys'       = physical event weight -> use for training & yields
    df["w_phys"]  = df["weight"] * df["weight_final"]
    df["label"]   = label   # 1 = signal, 0 = background
    df["process"] = proc    # keep track of the originating process
    df["run"]     = RUN
    dfs.append(df)
    print(f"{proc:12s}: {len(df):>8d} events after preselection")

data = pd.concat(dfs, ignore_index=True)

# ---- Post-concat cleaning --------------------------------------------------
# 1) Constant / empty features (zero variance, all-NaN or all-sentinel) -> drop
nun = data[features].nunique()
const = nun[nun <= 1].index.tolist()
features = [f for f in features if f not in const]
data = data.drop(columns=const)
print(f"\nDropped {len(const)} constant/empty features:\n  {sorted(const)}")

# 2) Sentinel values (e.g. -999) -> NaN; XGBoost routes NaNs natively.
#    Safe threshold here: all legit features are > -100 (eta, phi, pdg, charge).
for f in features:
    m = data[f] < -100
    if m.any():
        print(f"  sentinel -> NaN: {f} ({m.mean():.1%})")
        data[f] = data[f].mask(m)

print(f"\n{len(features)} final features")
print(f"Total: {len(data)} events | signal = {(data.label==1).sum()} | background = {(data.label==0).sum()}")
print(f"Yield (w_phys): signal = {data.loc[data.label==1,'w_phys'].sum():.2f} | background = {data.loc[data.label==0,'w_phys'].sum():.2f}")
data.head()


93 features kept (leakage-free) | weight = weight * weight_final

signal_ggF  :    67075 events after preselection
signal_VBF  :    21639 events after preselection
Diboson     :   312600 events after preselection
Zjets       :   141863 events after preselection
Wjets       :    31272 events after preselection
ttbar       :     7335 events after preselection
tops        :    62238 events after preselection
SingleH     :    11124 events after preselection
Vgamma      :    21510 events after preselection
VVV         :    14028 events after preselection

Total: 690684 events | signal = 88714 | background = 601970
Yield (sum of physical weights): signal = 1.82 | background = 11467.29


,n_jet,met_met,met_phi,met_sumet,pass_SLT,pass_DLT,pass_STT,pass_LTT,pass_DTT,l1_pdg,...,SumPt_t1t2,SumPt_l1j,SumPt_l1j1j2,SumPt_t1t2l1,weight,weight_norm,weight_gen,label,process,run
0,5,48261.832031,2.888055,479146.93750,1,0,0,0,0,-11,...,12270.166016,77389.632812,NaN,NaN,-0.000036,0.00125,-0.028450,1,signal_ggF,2
1,4,40528.195312,-2.665812,225360.81250,1,0,0,1,1,11,...,20957.582031,35689.703125,NaN,NaN,0.000069,0.00125,0.054811,1,signal_ggF,2
2,2,54573.648438,3.030940,364811.71875,1,0,0,1,1,11,...,33988.949219,127177.953125,NaN,NaN,0.000037,0.00125,0.029853,1,signal_ggF,2
3,3,95455.531250,-2.255971,329051.43750,0,0,0,1,0,11,...,18972.347656,36656.707031,NaN,NaN,0.000039,0.00125,0.030944,1,signal_ggF,2
4,3,55639.507812,-2.987233,354139.28125,1,0,0,1,1,-11,...,55308.082031,129401.234375,NaN,NaN,0.000041,0.00125,0.032416,1,signal_ggF,2


In [15]:
print(data.shape)

(690684, 77)


In [24]:
from sklearn.model_selection import train_test_split

# 80 / 20 train-validation split for Run 2, keeping the signal/background balance
train_df, val_df = train_test_split(
    data,
    test_size=0.20,
    random_state=42,
    stratify=data["label"],
)

X_train = train_df[features]
y_train = train_df["label"]
w_train = train_df["weight"]

X_val = val_df[features]
y_val = val_df["label"]
w_val = val_df["weight"]

print(f"Train: {len(train_df)} events | signal = {(y_train==1).sum()} | background = {(y_train==0).sum()}")
print(f"Val:   {len(val_df)} events | signal = {(y_val==1).sum()} | background = {(y_val==0).sum()}")


Train: 552547 events | signal = 70971 | background = 481576
Val:   138137 events | signal = 17743 | background = 120394


## XGBoost

## Weights

In [8]:
for name in ["weight", "weights", "weight_final"]:   # ajustá a los nombres exactos
    try:
        w = tree[name].array(library="np")
        print(f"{name:15s}  suma={w.sum():.4g}  media={w.mean():.4g}  neg={(w<0).mean():.3f}")
    except Exception as e:
        print(f"{name:15s}  ERROR: {e}")

weight           suma=0.007813  media=2.534e-08  neg=0.000
weights          suma=7.724e+11  media=2.505e+06  neg=0.154
weight_final     suma=7.724e+11  media=2.505e+06  neg=0.154


In [9]:
w_norm  = tree["weight"].array(library="np")
w_gen   = tree["weight_final"].array(library="np")
w_total = w_norm * w_gen
print("suma:", w_total.sum(), "media:", w_total.mean(), "neg:", (w_total<0).mean())

suma: 20845.893134293878 media: 0.0676167084587615 neg: 0.15408618368770172


In [10]:
print(np.unique(w_norm).size, np.unique(w_norm)[:5])

54 [7.66539258e-09 7.85560044e-09 8.05634116e-09 8.51140018e-09
 8.85011554e-09]


In [12]:
print([k for k in tree.keys() if any(s in k.lower() for s in
       ["dsid", "sample", "mc_channel", "run", "period", "campaign"])])

['dsid']


In [19]:
import pandas as pd, numpy as np

dsid = tree["dsid"].array(library="np")
df = pd.DataFrame({"dsid": dsid, "w_norm": w_norm})

print("N DSIDs:", df.dsid.nunique())
print(df.groupby("dsid").w_norm.nunique())   # ¿cuántos valores de weight por DSID?        # ¿cuántos DSIDs?

N DSIDs: 18
dsid
700320    3
700321    3
700322    3
700323    3
700324    3
700325    3
700467    3
700468    3
700469    3
700470    3
700471    3
700472    3
700792    3
700793    3
700794    3
700901    3
700902    3
700903    3
Name: w_norm, dtype: int64


In [21]:
import uproot, numpy as np

path = "PPSSP_2026/1l2tau/run2/Zjets.root:AnalysisMiniTree"
tree = uproot.open(path)

arr = tree.arrays(
    ["weight", "weight_final", "weights", "weights_mc",
     "weights_detector", "weights_eff", "weight_ff"],
    library="np"
)

# 1) ¿weight_final es el producto de las tres?
prod = arr["weights_mc"] * arr["weights_detector"] * arr["weights_eff"]
print("weight_final == mc*det*eff :", np.allclose(prod, arr["weight_final"], rtol=1e-4))

# 2) ¿weights es duplicado de weight_final?
print("weights == weight_final    :", np.allclose(arr["weights"], arr["weight_final"], rtol=1e-4))

# 3) ¿Qué es weight_ff?
ff = arr["weight_ff"]
print("\nweight_ff:")
print("  únicos:", np.unique(ff)[:10])
print("  n únicos:", np.unique(ff).size)
print("  media:", ff.mean())
print("  fracción == 1:", (ff == 1).mean())
print("  fracción == 0:", (ff == 0).mean())

weight_final == mc*det*eff : True
weights == weight_final    : True

weight_ff:
  únicos: [1.]
  n únicos: 1
  media: 1.0
  fracción == 1: 1.0
  fracción == 0: 0.0
